# LearnTrack LMS Analytics

## 1. Project Overview

This project implements a batch data engineering and analytics pipeline for a Learning Management System (LMS). The pipeline processes learner, course, and enrolment data through Bronze, Silver, and Gold layers using PySpark and Delta Lake.

The objective is to transform raw LMS data into clean, enriched, and business-ready datasets that can be used to analyze learner engagement, course completion, instructor effectiveness, assessment performance, dropout behavior, and re-enrolment patterns.

## 2. Environment Setup

The project is implemented in Google Colab using PySpark and Delta Lake.

### Technologies Used

- Python
- Apache Spark / PySpark
- Delta Lake
- Pandas
- Google Colab

The Spark environment is configured to support distributed data processing and Delta Lake storage for the Bronze, Silver, and Gold layers.

In [4]:
!pip install pyspark delta-spark

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("LearnTrack_LMS_Analytics")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session Created Successfully!")

Spark Version: 4.0.3
Spark Session Created Successfully!


## 3. Data Sources and Initial Loading

The LMS dataset consists of learner, course, and enrolment activity data. The source datasets were loaded into the Spark environment for processing through the Bronze, Silver, and Gold data pipeline.

The initial loading stage preserves the source data and provides the foundation for subsequent data-quality checks, cleaning, enrichment, and analytical transformations.

In [6]:
learners_df = spark.read.csv(
    "learners.csv",
    header=True,
    inferSchema=True
)

learners_df.show(5)

+----------+----------------+--------------------+------------+-------+-----------------+-----------------+
|learner_id|    learner_name|               email|phone_number|   city|registration_date|subscription_type|
+----------+----------------+--------------------+------------+-------+-----------------+-----------------+
|   LRN0001|   Ananya Sharma|ananya.sharma.1@g...|  9433218196| Mumbai|       2022-04-06|             Free|
|   LRN0002|   Sachin Pillai|sachin.pillai.2@g...|  9083863794| Indore|       2022-06-13|          Premium|
|   LRN0003|   Suresh Mishra|suresh.mishra.3@r...|  9235116155|Chennai|       2022-02-14|          Premium|
|   LRN0004|     Vijay Kumar|vijay.kumar.4@gma...|  9618495931|  Surat|       2022-08-22|          Premium|
|   LRN0005|Priya Chatterjee|priya.chatterjee....|  9316475255|  Surat|       2022-10-01|          Premium|
+----------+----------------+--------------------+------------+-------+-----------------+-----------------+
only showing top 5 rows


In [7]:
learners_df.printSchema()

root
 |-- learner_id: string (nullable = true)
 |-- learner_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)



In [8]:
courses_df = spark.read.csv(
    "courses.csv",
    header=True,
    inferSchema=True
)

courses_df.show(5)

+---------+--------------------+---------------+-------------+---------------+--------------+----------------+---------+
|course_id|        course_title|       category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+--------------------+---------------+-------------+---------------+--------------+----------------+---------+
|   CRS001|Python for Data S...|   Data Science|       INS009|   Suresh Gupta|            15|    Intermediate|     2986|
|   CRS002|Machine Learning ...|        AI & ML|       INS006|    Priya Singh|            10|        Beginner|      223|
|   CRS003|Deep Learning wit...|        AI & ML|       INS004|     Sunita Rao|            10|        Advanced|    13612|
|   CRS004|React.js Complete...|Web Development|       INS014|   Pooja Mishra|            30|        Advanced|     6867|
|   CRS005|Node.js Backend D...|Web Development|       INS007|    Vikas Mehta|            20|        Beginner|      119|
+---------+--------------------+

In [9]:
courses_df.printSchema()

root
 |-- course_id: string (nullable = true)
 |-- course_title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- instructor_id: string (nullable = true)
 |-- instructor_name: string (nullable = true)
 |-- duration_hours: integer (nullable = true)
 |-- difficulty_level: string (nullable = true)
 |-- price_inr: integer (nullable = true)



In [10]:
enrolment_df = spark.read.csv(
    "enrolment_activity.csv",
    header=True,
    inferSchema=True
)

enrolment_df.show(5)

+------------+----------+---------+----------+------------------------+----------------------+---------+------------+------------------+----------------+--------+---------------+------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|   status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|
+------------+----------+---------+----------+------------------------+----------------------+---------+------------+------------------+----------------+--------+---------------+------------------+
|    ENR00460|   LRN0376|   CRS001|2024-01-13|              2024-02-12|            2024-02-21|Completed|         100|        2024-02-21|           79.22|       1|              2|               Yes|
|    ENR00681|   LRN0486|   CRS055|2024-01-16|              2024-01-26|            2024-01-19|Completed|         100|        2024-01-19|           56.62|       1|              3|                No|
|    ENR01

In [11]:
enrolment_df.printSchema()

root
 |-- enrolment_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)



In [12]:
print("Learners:", learners_df.count())
print("Courses:", courses_df.count())
print("Enrolment Activity:", enrolment_df.count())

Learners: 500
Courses: 60
Enrolment Activity: 2000


In [13]:
print("Learners Columns:")
print(learners_df.columns)

print("\nCourses Columns:")
print(courses_df.columns)

print("\nEnrolment Activity Columns:")
print(enrolment_df.columns)

Learners Columns:
['learner_id', 'learner_name', 'email', 'phone_number', 'city', 'registration_date', 'subscription_type']

Courses Columns:
['course_id', 'course_title', 'category', 'instructor_id', 'instructor_name', 'duration_hours', 'difficulty_level', 'price_inr']

Enrolment Activity Columns:
['enrolment_id', 'learner_id', 'course_id', 'enrol_date', 'expected_completion_date', 'actual_completion_date', 'status', 'progress_pct', 'last_activity_date', 'assessment_score', 'attempts', 'feedback_rating', 'certificate_issued']


In [14]:
print("===== LEARNERS =====")
learners_df.show(3, truncate=False)

print("===== COURSES =====")
courses_df.show(3, truncate=False)

print("===== ENROLMENT ACTIVITY =====")
enrolment_df.show(3, truncate=False)

===== LEARNERS =====
+----------+-------------+------------------------------+------------+-------+-----------------+-----------------+
|learner_id|learner_name |email                         |phone_number|city   |registration_date|subscription_type|
+----------+-------------+------------------------------+------------+-------+-----------------+-----------------+
|LRN0001   |Ananya Sharma|ananya.sharma.1@gmail.com     |9433218196  |Mumbai |2022-04-06       |Free             |
|LRN0002   |Sachin Pillai|sachin.pillai.2@gmail.com     |9083863794  |Indore |2022-06-13       |Premium          |
|LRN0003   |Suresh Mishra|suresh.mishra.3@rediffmail.com|9235116155  |Chennai|2022-02-14       |Premium          |
+----------+-------------+------------------------------+------------+-------+-----------------+-----------------+
only showing top 3 rows
===== COURSES =====
+---------+-----------------------------+------------+-------------+---------------+--------------+----------------+---------+
|co

## 4. Bronze Layer

The Bronze layer contains the raw LMS datasets ingested into Spark and stored in Delta format. The source data is preserved without applying business transformations. Initial data-quality checks are performed to understand null values, duplicate records, and other source-level issues before downstream processing.

### 4.1 Data Quality Checks

In [15]:
print("===== LEARNERS NULL COUNTS =====")
learners_df.select([
    __import__("pyspark").sql.functions.count(
        __import__("pyspark").sql.functions.when(
            __import__("pyspark").sql.functions.col(c).isNull(), c
        )
    ).alias(c)
    for c in learners_df.columns
]).show()

print("===== COURSES NULL COUNTS =====")
courses_df.select([
    __import__("pyspark").sql.functions.count(
        __import__("pyspark").sql.functions.when(
            __import__("pyspark").sql.functions.col(c).isNull(), c
        )
    ).alias(c)
    for c in courses_df.columns
]).show()

print("===== ENROLMENT NULL COUNTS =====")
enrolment_df.select([
    __import__("pyspark").sql.functions.count(
        __import__("pyspark").sql.functions.when(
            __import__("pyspark").sql.functions.col(c).isNull(), c
        )
    ).alias(c)
    for c in enrolment_df.columns
]).show()

===== LEARNERS NULL COUNTS =====
+----------+------------+-----+------------+----+-----------------+-----------------+
|learner_id|learner_name|email|phone_number|city|registration_date|subscription_type|
+----------+------------+-----+------------+----+-----------------+-----------------+
|         0|           0|    0|           0|   0|                0|                0|
+----------+------------+-----+------------+----+-----------------+-----------------+

===== COURSES NULL COUNTS =====
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|course_id|course_title|category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|        0|           0|       0|            0|              6|             0|               0|        0|
+---------+------------+--------+-------------+---------------+--------------+--

In [16]:
print("Total Enrolment Records:", enrolment_df.count())

print(
    "Duplicate Enrolment IDs:",
    enrolment_df.count() - enrolment_df.dropDuplicates(["enrolment_id"]).count()
)

print(
    "Duplicate Complete Rows:",
    enrolment_df.count() - enrolment_df.dropDuplicates().count()
)

Total Enrolment Records: 2000
Duplicate Enrolment IDs: 10
Duplicate Complete Rows: 10


In [17]:
from pyspark.sql.functions import col, trim, when, count

blank_counts = enrolment_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == ""),
            1
        )
    ).alias(c)
    for c in enrolment_df.columns
])

blank_counts.show()

+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+
|           0|         0|        0|         0|                       0|                  1179|     0|           0|               330|            1179|       0|           1082|                 0|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+



In [18]:
blank_course_counts = courses_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == ""),
            1
        )
    ).alias(c)
    for c in courses_df.columns
])

blank_course_counts.show()

+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|course_id|course_title|category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|        0|           0|       0|            0|              6|             0|               0|        0|
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+



In [19]:
blank_learner_counts = learners_df.select([
    count(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == ""),
            1
        )
    ).alias(c)
    for c in learners_df.columns
])

blank_learner_counts.show()

+----------+------------+-----+------------+----+-----------------+-----------------+
|learner_id|learner_name|email|phone_number|city|registration_date|subscription_type|
+----------+------------+-----+------------+----+-----------------+-----------------+
|         0|           0|    0|           0|   0|                0|                0|
+----------+------------+-----+------------+----+-----------------+-----------------+



In [20]:
bronze_path = "/content/bronze"

learners_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/learners")

courses_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/courses")

enrolment_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/enrolment_activity")

print("Bronze layer created successfully!")

Bronze layer created successfully!


In [21]:
print("Bronze Learners:")
spark.read.format("delta").load("/content/bronze/learners").show(3)

print("Bronze Courses:")
spark.read.format("delta").load("/content/bronze/courses").show(3)

print("Bronze Enrolment Activity:")
spark.read.format("delta").load("/content/bronze/enrolment_activity").show(3)

Bronze Learners:
+----------+-------------+--------------------+------------+-------+-----------------+-----------------+
|learner_id| learner_name|               email|phone_number|   city|registration_date|subscription_type|
+----------+-------------+--------------------+------------+-------+-----------------+-----------------+
|   LRN0001|Ananya Sharma|ananya.sharma.1@g...|  9433218196| Mumbai|       2022-04-06|             Free|
|   LRN0002|Sachin Pillai|sachin.pillai.2@g...|  9083863794| Indore|       2022-06-13|          Premium|
|   LRN0003|Suresh Mishra|suresh.mishra.3@r...|  9235116155|Chennai|       2022-02-14|          Premium|
+----------+-------------+--------------------+------------+-------+-----------------+-----------------+
only showing top 3 rows
Bronze Courses:
+---------+--------------------+------------+-------------+---------------+--------------+----------------+---------+
|course_id|        course_title|    category|instructor_id|instructor_name|duration_hours|

In [22]:
bronze_learners = spark.read.format("delta").load("/content/bronze/learners")
bronze_courses = spark.read.format("delta").load("/content/bronze/courses")
bronze_enrolment = spark.read.format("delta").load("/content/bronze/enrolment_activity")

print("Bronze Learners:", bronze_learners.count())
print("Bronze Courses:", bronze_courses.count())
print("Bronze Enrolment Activity:", bronze_enrolment.count())

Bronze Learners: 500
Bronze Courses: 60
Bronze Enrolment Activity: 2000


## 5. Silver Layer

The Silver layer applies data-quality transformations and business enrichment to the Bronze datasets. Duplicate enrolment records are removed, missing instructor names are resolved using instructor identifiers, date fields are standardized, learning duration is derived, and learner, course, and enrolment data are joined into an enriched dataset.

In [23]:
silver_learners = spark.read.format("delta").load("/content/bronze/learners")
silver_courses = spark.read.format("delta").load("/content/bronze/courses")
silver_enrolment = spark.read.format("delta").load("/content/bronze/enrolment_activity")

print("Silver processing DataFrames loaded successfully!")

Silver processing DataFrames loaded successfully!


### 5.1 Deduplication

In [24]:
silver_enrolment_dedup = silver_enrolment.dropDuplicates(["enrolment_id"])

print("Before Deduplication:", silver_enrolment.count())
print("After Deduplication :", silver_enrolment_dedup.count())
print("Records Removed     :",
      silver_enrolment.count() - silver_enrolment_dedup.count())

Before Deduplication: 2000
After Deduplication : 1990
Records Removed     : 10


### 5.2 Instructor Name Quality Check

In [25]:
silver_courses.filter(
    col("instructor_name").isNull() |
    (trim(col("instructor_name")) == "")
).select(
    "course_id",
    "course_title",
    "instructor_id",
    "instructor_name"
).show(truncate=False)

+---------+----------------------+-------------+---------------+
|course_id|course_title          |instructor_id|instructor_name|
+---------+----------------------+-------------+---------------+
|CRS006   |HTML & CSS Mastery    |INS015       |NULL           |
|CRS016   |Ethical Hacking Basics|INS010       |NULL           |
|CRS020   |SQL & Database Design |INS013       |NULL           |
|CRS038   |Reinforcement Learning|INS010       |NULL           |
|CRS042   |Operations Management |INS001       |NULL           |
|CRS060   |Bayesian Statistics   |INS006       |NULL           |
+---------+----------------------+-------------+---------------+



### 5.3 Instructor Name Resolution

In [26]:
from pyspark.sql.functions import col, trim, when, first

# Create instructor ID -> instructor name mapping
instructor_map = (
    silver_courses
    .filter(
        col("instructor_name").isNotNull() &
        (trim(col("instructor_name")) != "")
    )
    .groupBy("instructor_id")
    .agg(first("instructor_name").alias("mapped_instructor_name"))
)

# Join mapping back to courses and fill missing instructor names
silver_courses_clean = (
    silver_courses
    .join(instructor_map, on="instructor_id", how="left")
    .withColumn(
        "instructor_name",
        when(
            col("instructor_name").isNull() |
            (trim(col("instructor_name")) == ""),
            col("mapped_instructor_name")
        ).otherwise(col("instructor_name"))
    )
    .drop("mapped_instructor_name")
)

### 5.4 Instructor Mapping Validation

In [27]:
silver_courses_clean.filter(
    col("instructor_name").isNull() |
    (trim(col("instructor_name")) == "")
).count()

0

### 5.5 Date Standardization

In [28]:
from pyspark.sql.functions import to_date

silver_enrolment_dates = (
    silver_enrolment_dedup
    .withColumn("enrol_date", to_date("enrol_date"))
    .withColumn("expected_completion_date", to_date("expected_completion_date"))
    .withColumn("actual_completion_date", to_date("actual_completion_date"))
    .withColumn("last_activity_date", to_date("last_activity_date"))
)

silver_enrolment_dates.printSchema()

root
 |-- enrolment_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)



### 5.6 Learning Duration

In [29]:
from pyspark.sql.functions import datediff

silver_enriched = silver_enrolment_dates.withColumn(
    "learning_duration",
    datediff(
        "actual_completion_date",
        "enrol_date"
    )
)

silver_enriched.select(
    "enrolment_id",
    "enrol_date",
    "actual_completion_date",
    "learning_duration"
).show(10, truncate=False)

+------------+----------+----------------------+-----------------+
|enrolment_id|enrol_date|actual_completion_date|learning_duration|
+------------+----------+----------------------+-----------------+
|ENR00176    |2024-02-22|2024-03-19            |26               |
|ENR01757    |2024-03-09|NULL                  |NULL             |
|ENR00166    |2024-01-15|2024-01-23            |8                |
|ENR00141    |2024-01-30|NULL                  |NULL             |
|ENR01449    |2024-02-13|NULL                  |NULL             |
|ENR01860    |2024-03-11|2024-03-28            |17               |
|ENR00637    |2024-02-19|2024-03-21            |31               |
|ENR01361    |2024-02-20|2024-03-06            |15               |
|ENR01520    |2024-01-02|NULL                  |NULL             |
|ENR00335    |2024-02-11|2024-03-22            |40               |
+------------+----------+----------------------+-----------------+
only showing top 10 rows


### 5.7 Learner Enrichment

In [30]:
silver_learner_enrolment = (
    silver_enriched
    .join(
        silver_learners,
        on="learner_id",
        how="left"
    )
)

print("Learner + Enrolment join completed!")
print("Rows:", silver_learner_enrolment.count())

Learner + Enrolment join completed!
Rows: 1990


### 5.8 Course Enrichment

In [31]:
silver_full = (
    silver_learner_enrolment
    .join(
        silver_courses_clean,
        on="course_id",
        how="left"
    )
)

print("Complete Silver Join Successful!")
print("Rows:", silver_full.count())

Complete Silver Join Successful!
Rows: 1990


### 5.9 Silver Schema Validation

In [32]:
silver_full.printSchema()

root
 |-- course_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- enrolment_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)
 |-- learning_duration: integer (nullable = true)
 |-- learner_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- instructor_id: string (nullable = true)
 |-- course_title: string (nullable = true)
 |-- categ

### 5.10 Silver Record Validation

In [33]:
print("Final Silver Records:", silver_full.count())

Final Silver Records: 1990


### 5.11 Silver Delta Storage

In [34]:
silver_path = "/content/silver/enriched_enrolments"

silver_full.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Silver Delta Table created successfully!")

Silver Delta Table created successfully!


### 5.12 Silver Delta Validation

In [35]:
silver_table = (
    spark.read
    .format("delta")
    .load("/content/silver/enriched_enrolments")
)

print("Silver Records:", silver_table.count())

silver_table.show(5, truncate=False)

Silver Records: 1990
+---------+----------+------------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+-----------------+----------------+-------------------------------+------------+----------+-----------------+-----------------+-------------+-------------------------+---------------+---------------+--------------+----------------+---------+
|course_id|learner_id|enrolment_id|enrol_date|expected_completion_date|actual_completion_date|status     |progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration|learner_name    |email                          |phone_number|city      |registration_date|subscription_type|instructor_id|course_title             |category       |instructor_name|duration_hours|difficulty_level|price_inr|
+---------+----------+------------+----------+------------------------+----------------------+---

## 6. Gold Layer

The Gold layer contains business-ready analytical datasets derived from the enriched Silver layer. The outputs are designed to answer operational questions related to course completion, instructor effectiveness, learner engagement, assessment performance, dropout behavior, re-enrolment, category performance, and overall LMS KPIs.

In [36]:
from pyspark.sql.functions import count, when, round

gold_course_completion = (
    silver_table
    .groupBy(
        "course_id",
        "course_title",
        "category"
    )
    .agg(
        count("*").alias("total_enrolments"),
        count(
            when(col("status") == "Completed", True)
        ).alias("completed_enrolments")
    )
    .withColumn(
        "completion_rate",
        round(
            col("completed_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
)

gold_course_completion.show(10, truncate=False)

+---------+-----------------------------+------------------+----------------+--------------------+---------------+
|course_id|course_title                 |category          |total_enrolments|completed_enrolments|completion_rate|
+---------+-----------------------------+------------------+----------------+--------------------+---------------+
|CRS018   |Flutter App Development      |Mobile Development|35              |12                  |34.29          |
|CRS024   |Computer Vision              |AI & ML           |40              |18                  |45.0           |
|CRS011   |Figma for Beginners          |Design            |23              |13                  |56.52          |
|CRS036   |Advanced Python OOP          |Data Science      |22              |7                   |31.82          |
|CRS053   |Supply Chain Management      |Business          |30              |16                  |53.33          |
|CRS026   |Full Stack Django            |Web Development   |39              |11 

### 6.1 Course Completion Analysis


In [37]:
gold_course_completion = (
    gold_course_completion
    .withColumn(
        "completion_category",
        when(col("completion_rate") >= 70, "High")
        .when(col("completion_rate") >= 50, "Moderate")
        .otherwise("At Risk")
    )
)

gold_course_completion.select(
    "course_id",
    "course_title",
    "completion_rate",
    "completion_category"
).show(10, truncate=False)

+---------+-----------------------------+---------------+-------------------+
|course_id|course_title                 |completion_rate|completion_category|
+---------+-----------------------------+---------------+-------------------+
|CRS018   |Flutter App Development      |34.29          |At Risk            |
|CRS024   |Computer Vision              |45.0           |At Risk            |
|CRS011   |Figma for Beginners          |56.52          |Moderate           |
|CRS036   |Advanced Python OOP          |31.82          |At Risk            |
|CRS053   |Supply Chain Management      |53.33          |Moderate           |
|CRS026   |Full Stack Django            |28.21          |At Risk            |
|CRS021   |Power BI & Tableau           |38.24          |At Risk            |
|CRS007   |Business Analytics Essentials|35.48          |At Risk            |
|CRS017   |Network Security Essentials  |32.43          |At Risk            |
|CRS043   |Leadership & Communication   |34.29          |At Risk

### 6.2 Course Completion Classification


In [38]:
gold_instructor_performance = (
    silver_table
    .groupBy(
        "instructor_id",
        "instructor_name"
    )
    .agg(
        count("*").alias("total_enrolments"),
        count(
            when(col("status") == "Completed", True)
        ).alias("completed_enrolments")
    )
    .withColumn(
        "completion_rate",
        round(
            col("completed_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
    .orderBy(col("completion_rate").desc())
)

gold_instructor_performance.show(10, truncate=False)

+-------------+---------------+----------------+--------------------+---------------+
|instructor_id|instructor_name|total_enrolments|completed_enrolments|completion_rate|
+-------------+---------------+----------------+--------------------+---------------+
|INS009       |Suresh Gupta   |170             |83                  |48.82          |
|INS002       |Meera Nair     |159             |75                  |47.17          |
|INS010       |Anjali Iyer    |176             |77                  |43.75          |
|INS001       |Rajiv Sharma   |60              |26                  |43.33          |
|INS007       |Vikas Mehta    |141             |61                  |43.26          |
|INS014       |Pooja Mishra   |65              |28                  |43.08          |
|INS005       |Deepak Joshi   |103             |44                  |42.72          |
|INS012       |Nisha Agarwal  |51              |21                  |41.18          |
|INS006       |Priya Singh    |250             |100   

### 6.3 Instructor Performance



In [39]:
gold_course_completion.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/course_completion")

gold_instructor_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/instructor_performance")

print("Course Completion and Instructor Performance Gold tables saved successfully!")

Course Completion and Instructor Performance Gold tables saved successfully!


### 6.4 Gold Storage – Course and Instructor Analytics


In [40]:
from pyspark.sql.functions import max as spark_max

gold_learner_engagement = (
    silver_table
    .groupBy(
        "learner_id",
        "learner_name"
    )
    .agg(
        count("*").alias("total_enrolments"),
        spark_max("last_activity_date").alias("last_activity_date"),
        round(
            spark_max("progress_pct"),
            2
        ).alias("latest_progress_pct")
    )
)

gold_learner_engagement.show(10, truncate=False)

+----------+----------------+----------------+------------------+-------------------+
|learner_id|learner_name    |total_enrolments|last_activity_date|latest_progress_pct|
+----------+----------------+----------------+------------------+-------------------+
|LRN0267   |Sunita Gupta    |4               |2024-03-31        |100                |
|LRN0172   |Ajay Yadav      |6               |2024-03-04        |100                |
|LRN0386   |Deepak Kapoor   |4               |2024-03-30        |100                |
|LRN0222   |Neha Mukherjee  |4               |2024-03-31        |100                |
|LRN0464   |Komal Yadav     |2               |2024-01-29        |100                |
|LRN0038   |Ajay Verma      |4               |2024-03-31        |100                |
|LRN0398   |Karan Srivastava|2               |2024-03-23        |73                 |
|LRN0431   |Shreya Pillai   |6               |2024-03-25        |100                |
|LRN0309   |Simran Mehta    |3               |2024-03-

### 6.5 Learner Engagement



In [41]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

engagement_window = Window.partitionBy(
    "learner_id"
).orderBy(
    col("last_activity_date").desc_nulls_last()
)

learner_latest_activity = (
    silver_table
    .withColumn(
        "activity_rank",
        row_number().over(engagement_window)
    )
    .filter(col("activity_rank") == 1)
    .select(
        "learner_id",
        "learner_name",
        "last_activity_date",
        "progress_pct",
        "status"
    )
)

learner_latest_activity.show(10, truncate=False)

+----------+----------------+------------------+------------+-----------+
|learner_id|learner_name    |last_activity_date|progress_pct|status     |
+----------+----------------+------------------+------------+-----------+
|LRN0001   |Ananya Sharma   |2024-03-24        |90          |In Progress|
|LRN0002   |Sachin Pillai   |2024-03-02        |24          |Dropped    |
|LRN0003   |Suresh Mishra   |2024-03-03        |15          |Dropped    |
|LRN0004   |Vijay Kumar     |2024-03-28        |100         |Completed  |
|LRN0005   |Priya Chatterjee|2024-03-01        |77          |In Progress|
|LRN0006   |Karan Pillai    |2024-03-20        |13          |In Progress|
|LRN0007   |Aditya Bose     |2024-03-23        |100         |Completed  |
|LRN0008   |Divya Yadav     |2024-03-12        |0           |Dropped    |
|LRN0009   |Rohan Chatterjee|2024-02-16        |71          |In Progress|
|LRN0010   |Aarav Desai     |2024-03-31        |17          |In Progress|
+----------+----------------+---------

### 6.6 Latest Learner Activity


In [42]:
gold_learner_engagement = (
    learner_latest_activity
    .withColumn(
        "engagement_category",
        when(col("status") == "Dropped", "Dropped")
        .when(col("status") == "Completed", "Completed")
        .when(
            (col("status") == "In Progress") &
            (col("progress_pct") < 50),
            "Low Engagement"
        )
        .otherwise("Active")
    )
)

gold_learner_engagement.show(10, truncate=False)

+----------+----------------+------------------+------------+-----------+-------------------+
|learner_id|learner_name    |last_activity_date|progress_pct|status     |engagement_category|
+----------+----------------+------------------+------------+-----------+-------------------+
|LRN0001   |Ananya Sharma   |2024-03-24        |90          |In Progress|Active             |
|LRN0002   |Sachin Pillai   |2024-03-02        |24          |Dropped    |Dropped            |
|LRN0003   |Suresh Mishra   |2024-03-03        |15          |Dropped    |Dropped            |
|LRN0004   |Vijay Kumar     |2024-03-28        |100         |Completed  |Completed          |
|LRN0005   |Priya Chatterjee|2024-03-01        |77          |In Progress|Active             |
|LRN0006   |Karan Pillai    |2024-03-20        |13          |In Progress|Low Engagement     |
|LRN0007   |Aditya Bose     |2024-03-23        |100         |Completed  |Completed          |
|LRN0008   |Divya Yadav     |2024-03-12        |0           

### 6.7 Learner Engagement Classification


In [43]:
gold_learner_engagement.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/learner_engagement")

print("Learner Engagement Gold table saved successfully!")

Learner Engagement Gold table saved successfully!


### 6.8 Learner Engagement Gold Storage


In [44]:
from pyspark.sql.functions import avg, sum as spark_sum

gold_assessment_performance = (
    silver_table
    .filter(col("assessment_score").isNotNull())
    .groupBy(
        "course_id",
        "course_title",
        "category"
    )
    .agg(
        round(
            avg("assessment_score"),
            2
        ).alias("average_score"),
        spark_sum("attempts").alias("total_attempts"),
        count("assessment_score").alias("assessed_learners")
    )
    .orderBy(
        col("average_score").desc()
    )
)

gold_assessment_performance.show(10, truncate=False)

+---------+---------------------------+---------------+-------------+--------------+-----------------+
|course_id|course_title               |category       |average_score|total_attempts|assessed_learners|
+---------+---------------------------+---------------+-------------+--------------+-----------------+
|CRS058   |Zero Trust Security        |Cybersecurity  |81.61        |15            |10               |
|CRS004   |React.js Complete Guide    |Web Development|79.76        |21            |18               |
|CRS006   |HTML & CSS Mastery         |Web Development|77.98        |15            |12               |
|CRS028   |Entrepreneurship 101       |Business       |77.55        |20            |16               |
|CRS022   |Statistics for Data Science|Data Science   |76.31        |11            |9                |
|CRS011   |Figma for Beginners        |Design         |75.12        |17            |13               |
|CRS036   |Advanced Python OOP        |Data Science   |74.95        |12  

### 6.9 Assessment Performance


In [45]:
gold_assessment_performance = (
    gold_assessment_performance
    .withColumn(
        "assessment_category",
        when(col("average_score") >= 70, "Good")
        .when(col("average_score") >= 50, "Needs Attention")
        .otherwise("Bottleneck")
    )
)

gold_assessment_performance.select(
    "course_id",
    "course_title",
    "average_score",
    "total_attempts",
    "assessment_category"
).show(10, truncate=False)

+---------+---------------------------+-------------+--------------+-------------------+
|course_id|course_title               |average_score|total_attempts|assessment_category|
+---------+---------------------------+-------------+--------------+-------------------+
|CRS058   |Zero Trust Security        |81.61        |15            |Good               |
|CRS004   |React.js Complete Guide    |79.76        |21            |Good               |
|CRS006   |HTML & CSS Mastery         |77.98        |15            |Good               |
|CRS028   |Entrepreneurship 101       |77.55        |20            |Good               |
|CRS022   |Statistics for Data Science|76.31        |11            |Good               |
|CRS011   |Figma for Beginners        |75.12        |17            |Good               |
|CRS036   |Advanced Python OOP        |74.95        |12            |Good               |
|CRS055   |Motion UI Design           |74.72        |27            |Good               |
|CRS032   |DevOps CI/

### 6.10 Assessment Classification


In [46]:
gold_assessment_performance.groupBy(
    "assessment_category"
).count().orderBy(
    col("count").desc()
).show()

+-------------------+-----+
|assessment_category|count|
+-------------------+-----+
|               Good|   35|
|    Needs Attention|   25|
+-------------------+-----+



### 6.11 Assessment Gold Storage


In [47]:
gold_assessment_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/assessment_performance")

print("Assessment Performance Gold table saved successfully!")

Assessment Performance Gold table saved successfully!


### 6.12 Re-enrolment Analysis


In [48]:
gold_reenrolment = (
    silver_table
    .groupBy(
        "learner_id",
        "learner_name",
        "course_id",
        "course_title"
    )
    .agg(
        count("*").alias("enrolment_count"),
        spark_max("attempts").alias("max_attempts")
    )
    .filter(
        (col("enrolment_count") > 1) |
        (col("max_attempts") >= 2)
    )
    .orderBy(
        col("enrolment_count").desc()
    )
)

gold_reenrolment.show(10, truncate=False)

+----------+--------------+---------+-----------------------------+---------------+------------+
|learner_id|learner_name  |course_id|course_title                 |enrolment_count|max_attempts|
+----------+--------------+---------+-----------------------------+---------------+------------+
|LRN0298   |Sunita Agarwal|CRS046   |Terraform Infrastructure     |2              |2           |
|LRN0133   |Ravi Jain     |CRS055   |Motion UI Design             |2              |2           |
|LRN0275   |Manish Reddy  |CRS002   |Machine Learning Fundamentals|2              |2           |
|LRN0352   |Ajay Bose     |CRS044   |Adobe Illustrator Mastery    |2              |2           |
|LRN0087   |Ishaan Shah   |CRS051   |MLOps & Model Deployment     |2              |1           |
|LRN0036   |Megha Iyer    |CRS018   |Flutter App Development      |2              |1           |
|LRN0046   |Pooja Patel   |CRS013   |AWS Solutions Architect      |2              |1           |
|LRN0100   |Ritika Mehta  |CRS

### 6.13 Re-enrolment Reason Classification


In [49]:
print(
    "Total Re-enrolment Candidates:",
    gold_reenrolment.count()
)

Total Re-enrolment Candidates: 460


### 6.14 Re-enrolment Gold Storage


In [50]:
gold_reenrolment = (
    gold_reenrolment
    .withColumn(
        "reenrolment_reason",
        when(
            (col("enrolment_count") > 1) &
            (col("max_attempts") >= 2),
            "Multiple Enrolments + Multiple Attempts"
        )
        .when(
            col("enrolment_count") > 1,
            "Multiple Enrolments"
        )
        .otherwise(
            "Multiple Attempts"
        )
    )
)

gold_reenrolment.groupBy(
    "reenrolment_reason"
).count().orderBy(
    col("count").desc()
).show()

+--------------------+-----+
|  reenrolment_reason|count|
+--------------------+-----+
|   Multiple Attempts|  380|
| Multiple Enrolments|   49|
|Multiple Enrolmen...|   31|
+--------------------+-----+



### 6.15 Dropout Analysis


In [51]:
gold_reenrolment.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/reenrolment_analysis")

print("Re-enrolment Gold table saved successfully!")

Re-enrolment Gold table saved successfully!


### 6.16 Overall Dropout KPI



In [52]:
gold_dropout = (
    silver_table
    .groupBy(
        "course_id",
        "course_title",
        "category"
    )
    .agg(
        count("*").alias("total_enrolments"),
        count(
            when(col("status") == "Dropped", True)
        ).alias("dropped_enrolments")
    )
    .withColumn(
        "dropout_rate",
        round(
            col("dropped_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
    .orderBy(
        col("dropout_rate").desc()
    )
)

gold_dropout.show(10, truncate=False)

+---------+-----------------------------+---------------+----------------+------------------+------------+
|course_id|course_title                 |category       |total_enrolments|dropped_enrolments|dropout_rate|
+---------+-----------------------------+---------------+----------------+------------------+------------+
|CRS043   |Leadership & Communication   |Business       |35              |12                |34.29       |
|CRS030   |3D Modelling with Blender    |Design         |30              |10                |33.33       |
|CRS021   |Power BI & Tableau           |Data Science   |34              |11                |32.35       |
|CRS012   |Motion Graphics & Animation  |Design         |32              |10                |31.25       |
|CRS052   |Svelte & SvelteKit           |Web Development|36              |11                |30.56       |
|CRS003   |Deep Learning with TensorFlow|AI & ML        |37              |11                |29.73       |
|CRS056   |NoSQL & MongoDB           

### 6.17 Dropout Gold Storage


In [53]:
total_enrolments = silver_table.count()

total_dropped = (
    silver_table
    .filter(col("status") == "Dropped")
    .count()
)

overall_dropout_rate = (
    total_dropped / total_enrolments
) * 100

overall_dropout_rate = float(
    f"{overall_dropout_rate:.2f}"
)

print("Total Enrolments:", total_enrolments)
print("Total Dropped:", total_dropped)
print("Overall Dropout Rate:", overall_dropout_rate, "%")

Total Enrolments: 1990
Total Dropped: 411
Overall Dropout Rate: 20.65 %


In [54]:
gold_dropout.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/dropout_analysis")

print("Dropout Gold table saved successfully!")

Dropout Gold table saved successfully!


### 6.18 Category Performance


In [55]:
gold_category_performance = (
    silver_table
    .groupBy("category")
    .agg(
        count("*").alias("total_enrolments"),
        count(
            when(col("status") == "Completed", True)
        ).alias("completed_enrolments"),
        count(
            when(col("status") == "Dropped", True)
        ).alias("dropped_enrolments")
    )
    .withColumn(
        "completion_rate",
        round(
            col("completed_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
    .withColumn(
        "dropout_rate",
        round(
            col("dropped_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
    .orderBy(
        col("completion_rate").desc()
    )
)

gold_category_performance.show(
    truncate=False
)

+------------------+----------------+--------------------+------------------+---------------+------------+
|category          |total_enrolments|completed_enrolments|dropped_enrolments|completion_rate|dropout_rate|
+------------------+----------------+--------------------+------------------+---------------+------------+
|Mobile Development|184             |91                  |29                |49.46          |15.76       |
|AI & ML           |253             |114                 |51                |45.06          |20.16       |
|Design            |249             |109                 |55                |43.78          |22.09       |
|Data Science      |291             |117                 |68                |40.21          |23.37       |
|Business          |300             |119                 |63                |39.67          |21.0        |
|Web Development   |290             |115                 |62                |39.66          |21.38       |
|Cloud Computing   |231             |

### 6.19 Category Gold Storage


In [56]:
gold_category_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/category_performance")

print("Category Performance Gold table saved successfully!")

Category Performance Gold table saved successfully!


### 6.20 Enhanced Instructor Effectiveness


In [57]:
gold_instructor_performance = (
    silver_table
    .groupBy(
        "instructor_id",
        "instructor_name"
    )
    .agg(
        count("*").alias("total_enrolments"),

        count(
            when(col("status") == "Completed", True)
        ).alias("completed_enrolments"),

        round(
            avg("assessment_score"),
            2
        ).alias("average_assessment_score"),

        spark_sum("attempts").alias("total_attempts")
    )
    .withColumn(
        "completion_rate",
        round(
            col("completed_enrolments") /
            col("total_enrolments") * 100,
            2
        )
    )
    .select(
        "instructor_id",
        "instructor_name",
        "total_enrolments",
        "completed_enrolments",
        "completion_rate",
        "average_assessment_score",
        "total_attempts"
    )
    .orderBy(
        col("completion_rate").desc()
    )
)

gold_instructor_performance.show(15, truncate=False)

+-------------+---------------+----------------+--------------------+---------------+------------------------+--------------+
|instructor_id|instructor_name|total_enrolments|completed_enrolments|completion_rate|average_assessment_score|total_attempts|
+-------------+---------------+----------------+--------------------+---------------+------------------------+--------------+
|INS009       |Suresh Gupta   |170             |83                  |48.82          |74.01                   |207           |
|INS002       |Meera Nair     |159             |75                  |47.17          |70.03                   |191           |
|INS010       |Anjali Iyer    |176             |77                  |43.75          |69.27                   |221           |
|INS001       |Rajiv Sharma   |60              |26                  |43.33          |69.52                   |79            |
|INS007       |Vikas Mehta    |141             |61                  |43.26          |67.97                   |174     

### 6.21 Instructor Performance Gold Storage

In [58]:
gold_instructor_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/content/gold/instructor_performance")

print("Updated Instructor Performance Gold table saved successfully!")

Updated Instructor Performance Gold table saved successfully!


### 6.22 Learner Progress Distribution

In [59]:
gold_progress_distribution = (
    gold_learner_engagement
    .withColumn(
        "progress_category",
        when(col("progress_pct") < 25, "Very Low")
        .when(col("progress_pct") < 50, "Low")
        .when(col("progress_pct") < 75, "Moderate")
        .otherwise("High")
    )
)

gold_progress_distribution.groupBy(
    "progress_category"
).count().orderBy(
    col("count").desc()
).show()

+-----------------+-----+
|progress_category|count|
+-----------------+-----+
|             High|  290|
|         Very Low|   98|
|              Low|   64|
|         Moderate|   43|
+-----------------+-----+



### 6.23 Progress Distribution Gold Storage


In [60]:
gold_progress_distribution.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/progress_distribution")

print("Progress Distribution Gold table saved successfully!")

Progress Distribution Gold table saved successfully!


### 6.24 Platform KPI Calculation


In [61]:
id="kpi2"
total_learners = silver_table.select("learner_id").distinct().count()
total_courses = silver_table.select("course_id").distinct().count()
total_enrolments = silver_table.count()

total_completed = (
    silver_table
    .filter(col("status") == "Completed")
    .count()
)

total_dropped = (
    silver_table
    .filter(col("status") == "Dropped")
    .count()
)

overall_completion_rate = float(
    f"{(total_completed / total_enrolments) * 100:.2f}"
)

overall_dropout_rate = float(
    f"{(total_dropped / total_enrolments) * 100:.2f}"
)

print("===== LMS GOLD KPI SUMMARY =====")
print("Total Learners:", total_learners)
print("Total Courses:", total_courses)
print("Total Enrolments:", total_enrolments)
print("Completed Enrolments:", total_completed)
print("Dropped Enrolments:", total_dropped)
print("Overall Completion Rate:", overall_completion_rate, "%")
print("Overall Dropout Rate:", overall_dropout_rate, "%")

===== LMS GOLD KPI SUMMARY =====
Total Learners: 495
Total Courses: 60
Total Enrolments: 1990
Completed Enrolments: 818
Dropped Enrolments: 411
Overall Completion Rate: 41.11 %
Overall Dropout Rate: 20.65 %


### 6.25 KPI DataFrame


In [62]:
kpi_data = [
    (
        total_learners,
        total_courses,
        total_enrolments,
        total_completed,
        total_dropped,
        overall_completion_rate,
        overall_dropout_rate
    )
]

gold_kpi_summary = spark.createDataFrame(
    kpi_data,
    [
        "total_learners",
        "total_courses",
        "total_enrolments",
        "completed_enrolments",
        "dropped_enrolments",
        "overall_completion_rate",
        "overall_dropout_rate"
    ]
)

gold_kpi_summary.show()

+--------------+-------------+----------------+--------------------+------------------+-----------------------+--------------------+
|total_learners|total_courses|total_enrolments|completed_enrolments|dropped_enrolments|overall_completion_rate|overall_dropout_rate|
+--------------+-------------+----------------+--------------------+------------------+-----------------------+--------------------+
|           495|           60|            1990|                 818|               411|                  41.11|               20.65|
+--------------+-------------+----------------+--------------------+------------------+-----------------------+--------------------+



### 6.26 KPI Gold Storage


In [63]:
gold_kpi_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/gold/kpi_summary")

print("Gold KPI Summary table saved successfully!")

Gold KPI Summary table saved successfully!


## 7. Gold Layer Validation

All Gold layer Delta datasets are read back from their storage paths and validated by checking that the generated tables can be successfully accessed and contain the expected analytical records.

In [64]:
gold_tables = {
    "Course Completion": "/content/gold/course_completion",
    "Instructor Performance": "/content/gold/instructor_performance",
    "Learner Engagement": "/content/gold/learner_engagement",
    "Assessment Performance": "/content/gold/assessment_performance",
    "Re-enrolment Analysis": "/content/gold/reenrolment_analysis",
    "Dropout Analysis": "/content/gold/dropout_analysis",
    "Category Performance": "/content/gold/category_performance",
    "Progress Distribution": "/content/gold/progress_distribution",
    "KPI Summary": "/content/gold/kpi_summary"
}

print("===== GOLD TABLE VALIDATION =====")

for table_name, table_path in gold_tables.items():
    df = spark.read.format("delta").load(table_path)
    print(f"{table_name}: {df.count()} records")

===== GOLD TABLE VALIDATION =====
Course Completion: 60 records
Instructor Performance: 14 records
Learner Engagement: 495 records
Assessment Performance: 60 records
Re-enrolment Analysis: 460 records
Dropout Analysis: 60 records
Category Performance: 8 records
Progress Distribution: 495 records
KPI Summary: 1 records


## 8. Business Insights

The Gold-layer analytical outputs are used to derive business insights related to learner engagement, course completion, instructor effectiveness, assessment performance, dropout behavior, re-enrolment patterns, category performance, and overall LMS KPIs.

# Final Business Insights Report

## 1. Platform Overview

The LMS analytics pipeline processed the cleaned learner, course, and enrolment data and generated business-ready analytical datasets in the Gold layer.

Key platform metrics:

- Total Learners: 495
- Total Courses: 60
- Total Enrolments: 1,990
- Completed Enrolments: 818
- Dropped Enrolments: 411
- Overall Completion Rate: 41.11%
- Overall Dropout Rate: 20.65%

## 2. Course Completion

Course-level completion analysis was performed using total enrolments and completed enrolments.

Courses were classified into High, Moderate, and At Risk categories based on completion rate.

The analysis helps identify courses where learner completion is relatively low and where additional learner support or course improvement may be required.

## 3. Instructor Effectiveness

Instructor performance was evaluated using multiple metrics rather than completion rate alone.

The Gold dataset contains:

- Total enrolments
- Completed enrolments
- Completion rate
- Average assessment score
- Total assessment attempts

This provides a broader view of instructor effectiveness by considering both learner completion and assessment outcomes.

## 4. Learner Engagement

The latest activity record for each learner was identified using a window function.

Learners were classified as:

- Active
- Low Engagement
- Completed
- Dropped

The latest activity and progress information can help administrators identify learners who may require additional engagement support.

## 5. Assessment Performance

Course-level assessment performance was analyzed using average assessment score, total attempts, and assessed learners.

Based on the analytical classification:

- Good: 35 courses
- Needs Attention: 25 courses
- Bottleneck: 0 courses

The results indicate that a significant number of courses may benefit from further assessment or instructional review even though no course fell into the Bottleneck category.

## 6. Dropout Analysis

A total of 411 enrolments were marked as Dropped out of 1,990 enrolments.

The overall dropout rate was:

20.65%

At the course level, Leadership & Communication recorded the highest displayed dropout rate of 34.29%.

Category-level analysis showed:

- Data Science had the highest dropout rate: 23.37%
- Mobile Development had the lowest dropout rate: 15.76%

## 7. Re-enrolment Analysis

The pipeline identified 460 learner-course combinations as re-enrolment candidates.

Breakdown:

- Multiple Attempts: 380
- Multiple Enrolments: 49
- Multiple Enrolments + Multiple Attempts: 31

Multiple assessment attempts represented the largest re-enrolment signal in the analyzed data.

## 8. Category Performance

Category-level analysis compared completion and dropout rates.

Mobile Development recorded the highest completion rate at 49.46%.

Cybersecurity recorded the lowest completion rate at 32.81%.

Data Science recorded the highest dropout rate at 23.37%.

## 9. Learner Progress Distribution

Latest learner progress was divided into analytical progress categories.

Distribution:

- High: 290 learners
- Moderate: 43 learners
- Low: 64 learners
- Very Low: 98 learners

This distribution can help identify learner groups that may require additional intervention.

## 10. Business Recommendations

Based on the generated analytics:

1. Investigate courses with comparatively low completion rates.
2. Review categories with higher dropout rates, particularly Data Science.
3. Provide additional support to learners classified as Low Engagement or Very Low progress.
4. Investigate courses classified as Needs Attention based on assessment performance.
5. Analyze repeated assessment attempts to identify potential learning difficulties.
6. Use instructor completion and assessment metrics together when evaluating instructional effectiveness.
7. Monitor re-enrolment patterns to understand recurring learner difficulties.

## 11. Final Outcome

The project successfully transforms raw LMS operational data into structured, cleaned, enriched, and business-ready analytical datasets using a Bronze-Silver-Gold architecture.

The Gold layer provides actionable metrics for learner engagement, course effectiveness, instructor performance, assessment outcomes, dropout behavior, and re-enrolment patterns.

## 9. Conclusion

The LearnTrack LMS Analytics pipeline successfully transforms raw LMS data into clean, enriched, and business-ready analytical datasets using a Bronze-Silver-Gold architecture.

The Bronze layer preserves the raw source data and performs initial data-quality checks. The Silver layer applies deduplication, instructor name resolution, date standardization, learning-duration calculation, and learner and course enrichment. The Gold layer then produces business-focused analytical datasets covering course completion, learner engagement, assessment performance, re-enrolment, dropout behavior, category performance, instructor effectiveness, learner progress, and overall platform KPIs.

The final analysis covers 495 learners, 60 courses, and 1,990 enrolments after removing 10 duplicate enrolment records. The overall completion rate is 41.11%, while the overall dropout rate is 20.65%.

The resulting Gold datasets provide a strong foundation for LMS performance monitoring and can support future reporting, dashboards, and deeper learner and course analysis.